# EV Intelligence Analytics - Market Analysis

## Overview
This notebook provides comprehensive market analysis including competitive landscape, market segmentation, pricing strategies, and business intelligence insights for strategic decision-making.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Load processed data
ev_sales = pd.read_csv('../data/processed/ev_sales_clean.csv')
charging_stations = pd.read_csv('../data/processed/charging_stations_clean.csv')
market_metrics = pd.read_csv('../data/processed/market_metrics_clean.csv')
manufacturer_performance = pd.read_csv('../data/processed/manufacturer_performance.csv')
merged_data = pd.read_csv('../data/processed/merged_analytics_data.csv')

# Convert date columns
ev_sales['date'] = pd.to_datetime(ev_sales['date'])
market_metrics['date'] = pd.to_datetime(market_metrics['date'])
merged_data['date'] = pd.to_datetime(merged_data['date'])

print("Datasets loaded successfully!")
print(f"EV Sales: {ev_sales.shape}")
print(f"Charging Stations: {charging_stations.shape}")
print(f"Market Metrics: {market_metrics.shape}")
print(f"Manufacturer Performance: {manufacturer_performance.shape}")
print(f"Merged Analytics Data: {merged_data.shape}")

## 1. Market Segmentation Analysis

In [ ]:
# State market segmentation
state_features = merged_data.groupby('state').agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'ev_penetration_rate': 'mean',
    'gdp_per_capita': 'mean',
    'total_stations': 'mean',
    'stations_per_1000_ev': 'mean',
    'ev_density': 'mean',
    'price_avg': 'mean'
}).reset_index()

# Prepare data for clustering
scaler = StandardScaler()
features_for_clustering = [
    'sales_amount', 'ev_penetration_rate', 'gdp_per_capita', 
    'total_stations', 'ev_density', 'price_avg'
]

X_scaled = scaler.fit_transform(state_features[features_for_clustering])

# Perform K-means clustering
kmeans = KMeans(n_clusters=4, random_state=42)
state_features['market_segment'] = kmeans.fit_predict(X_scaled)

# Analyze segments
segment_analysis = state_features.groupby('market_segment').agg({
    'state': 'count',
    'sales_amount': 'mean',
    'ev_penetration_rate': 'mean',
    'gdp_per_capita': 'mean',
    'total_stations': 'mean',
    'price_avg': 'mean'
}).round(2)

# Create segment visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Market Segments - Sales vs Penetration', 
                   'Market Segments - GDP vs Infrastructure',
                   'Segment Distribution', 'Segment Characteristics'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "pie"}, {"type": "bar"}]]
)

# Sales vs Penetration
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for segment in range(4):
    segment_data = state_features[state_features['market_segment'] == segment]
    fig.add_trace(
        go.Scatter(
            x=segment_data['ev_penetration_rate'],
            y=segment_data['sales_amount'],
            mode='markers',
            text=segment_data['state'],
            textposition="top center",
            marker=dict(size=12, color=colors[segment]),
            name=f'Segment {segment}'
        ),
        row=1, col=1
    )

# GDP vs Infrastructure
for segment in range(4):
    segment_data = state_features[state_features['market_segment'] == segment]
    fig.add_trace(
        go.Scatter(
            x=segment_data['gdp_per_capita'],
            y=segment_data['total_stations'],
            mode='markers',
            marker=dict(size=12, color=colors[segment]),
            name=f'Segment {segment}',
            showlegend=False
        ),
        row=1, col=2
    )

# Segment distribution
segment_counts = state_features['market_segment'].value_counts().sort_index()
fig.add_trace(
    go.Pie(
        labels=[f'Segment {i}' for i in segment_counts.index],
        values=segment_counts.values,
        name="Distribution"
    ),
    row=2, col=1
)

# Segment characteristics
fig.add_trace(
    go.Bar(
        x=[f'Segment {i}' for i in segment_analysis.index],
        y=segment_analysis['sales_amount'],
        name='Avg Sales',
        marker_color='#2ca02c'
    ),
    row=2, col=2
)

fig.update_layout(height=800, title_text="EV Market Segmentation Analysis")
fig.show()

# Print segment insights
print("=== MARKET SEGMENTATION INSIGHTS ===")
segment_names = ['Emerging Markets', 'Growth Markets', 'Mature Markets', 'Premium Markets']
for i, (segment, data) in enumerate(segment_analysis.iterrows()):
    print(f"\n{segment_names[i]} (Segment {segment}):")
    print(f"  States: {data['state']} states")
    print(f"  Avg Sales: {data['sales_amount']:,.0f} units")
    print(f"  Penetration Rate: {data['ev_penetration_rate']:.2%}")
    print(f"  GDP per Capita: ${data['gdp_per_capita']:,.0f}")
    print(f"  Avg Price: ${data['price_avg']:,.0f}")

## 2. Competitive Landscape Analysis

In [ ]:
# Manufacturer competitive analysis
manufacturer_competitive = ev_sales.groupby('manufacturer').agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'price_avg': 'mean',
    'battery_capacity': 'mean',
    'charging_time': 'mean',
    'market_share': 'mean',
    'state': 'nunique'
}).reset_index()

# Calculate competitive metrics
manufacturer_competitive['market_share_pct'] = (
    manufacturer_competitive['sales_amount'] / manufacturer_competitive['sales_amount'].sum()
) * 100

manufacturer_competitive['price_positioning'] = pd.cut(
    manufacturer_competitive['price_avg'],
    bins=3,
    labels=['Economy', 'Mid-Range', 'Premium']
)

manufacturer_competitive['tech_leadership'] = (
    manufacturer_competitive['battery_capacity'] / manufacturer_competitive['charging_time']
)

# Create competitive landscape dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Market Share vs Price Positioning', 
                   'Technology Leadership Matrix',
                   'Geographic Reach', 'Competitive Positioning'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "bar"}, {"type": "bubble"}]]
)

# Market Share vs Price Positioning
fig.add_trace(
    go.Scatter(
        x=manufacturer_competitive['price_avg'],
        y=manufacturer_competitive['market_share_pct'],
        mode='markers+text',
        text=manufacturer_competitive['manufacturer'],
        textposition="top center",
        marker=dict(
            size=manufacturer_competitive['sales_amount']/500,
            color=manufacturer_competitive['tech_leadership'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Tech Leadership")
        ),
        name='Manufacturers'
    ),
    row=1, col=1
)

# Technology Leadership Matrix
fig.add_trace(
    go.Scatter(
        x=manufacturer_competitive['battery_capacity'],
        y=manufacturer_competitive['tech_leadership'],
        mode='markers+text',
        text=manufacturer_competitive['manufacturer'],
        textposition="top center",
        marker=dict(
            size=manufacturer_competitive['market_share_pct'],
            color='#ff7f0e'
        ),
        name='Tech Leaders',
        showlegend=False
    ),
    row=1, col=2
)

# Geographic Reach
fig.add_trace(
    go.Bar(
        x=manufacturer_competitive['manufacturer'],
        y=manufacturer_competitive['state'],
        name='States Presence',
        marker_color='#2ca02c'
    ),
    row=2, col=1
)

# Competitive Positioning Bubble Chart
fig.add_trace(
    go.Scatter(
        x=manufacturer_competitive['price_avg'],
        y=manufacturer_competitive['market_share_pct'],
        mode='markers',
        marker=dict(
            size=manufacturer_competitive['tech_leadership']*20,
            color=manufacturer_competitive['revenue'],
            colorscale='RdYlBu',
            showscale=True,
            colorbar=dict(title="Revenue")
        ),
        text=manufacturer_competitive['manufacturer'],
        name='Positioning',
        showlegend=False
    ),
    row=2, col=2
)

fig.update_layout(height=800, title_text="EV Competitive Landscape Analysis")
fig.show()

# Competitive insights
print("=== COMPETITIVE LANDSCAPE INSIGHTS ===")
market_leader = manufacturer_competitive.loc[manufacturer_competitive['market_share_pct'].idxmax()]
tech_leader = manufacturer_competitive.loc[manufacturer_competitive['tech_leadership'].idxmax()]
premium_brand = manufacturer_competitive.loc[manufacturer_competitive['price_avg'].idxmax()]
widest_reach = manufacturer_competitive.loc[manufacturer_competitive['state'].idxmax()]

print(f"🏆 Market Leader: {market_leader['manufacturer']} ({market_leader['market_share_pct']:.1f}% market share)")
print(f"⚡ Technology Leader: {tech_leader['manufacturer']} (Tech Score: {tech_leader['tech_leadership']:.2f})")
print(f"💎 Premium Brand: {premium_brand['manufacturer']} (${premium_brand['price_avg']:,.0f} avg price)")
print(f"🌍 Widest Reach: {widest_reach['manufacturer']} ({widest_reach['state']} states)")

## 3. Pricing Strategy Analysis

In [ ]:
# Pricing analysis by category and manufacturer
pricing_analysis = ev_sales.groupby(['manufacturer', 'vehicle_category', 'vehicle_type']).agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'price_avg': 'mean',
    'price_min': 'mean',
    'price_max': 'mean',
    'battery_capacity': 'mean',
    'market_share': 'mean'
}).reset_index()

# Calculate price elasticity indicators
pricing_analysis['price_range_width'] = pricing_analysis['price_max'] - pricing_analysis['price_min']
pricing_analysis['price_per_kwh'] = pricing_analysis['price_avg'] / pricing_analysis['battery_capacity']
pricing_analysis['value_score'] = pricing_analysis['battery_capacity'] / pricing_analysis['price_avg'] * 1000

# Create pricing strategy dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Price Distribution by Category', 
                   'Price vs Volume Analysis',
                   'Value Proposition Matrix', 'Price per kWh Comparison'),
    specs=[[{"type": "box"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "bar"}]]
)

# Price distribution by category
categories = pricing_analysis['vehicle_category'].unique()
for i, category in enumerate(categories):
    category_data = pricing_analysis[pricing_analysis['vehicle_category'] == category]
    fig.add_trace(
        go.Box(
            y=category_data['price_avg'],
            name=category,
            marker_color=colors[i % len(colors)]
        ),
        row=1, col=1
    )

# Price vs Volume Analysis
fig.add_trace(
    go.Scatter(
        x=pricing_analysis['price_avg'],
        y=pricing_analysis['sales_amount'],
        mode='markers',
        text=pricing_analysis['manufacturer'] + ' ' + pricing_analysis['vehicle_type'],
        marker=dict(
            size=pricing_analysis['market_share']*100,
            color=pricing_analysis['value_score'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Value Score")
        ),
        name='Price-Volume'
    ),
    row=1, col=2
)

# Value Proposition Matrix
fig.add_trace(
    go.Scatter(
        x=pricing_analysis['price_avg'],
        y=pricing_analysis['value_score'],
        mode='markers+text',
        text=pricing_analysis['manufacturer'],
        textposition="top center",
        marker=dict(
            size=pricing_analysis['sales_amount']/100,
            color='#ff7f0e'
        ),
        name='Value Proposition',
        showlegend=False
    ),
    row=2, col=1
)

# Price per kWh by manufacturer
manufacturer_price_kwh = pricing_analysis.groupby('manufacturer')['price_per_kwh'].mean().sort_values()
fig.add_trace(
    go.Bar(
        x=manufacturer_price_kwh.index,
        y=manufacturer_price_kwh.values,
        name='Price per kWh',
        marker_color='#2ca02c'
    ),
    row=2, col=2
)

fig.update_layout(height=800, title_text="EV Pricing Strategy Analysis")
fig.show()

# Pricing insights
print("=== PRICING STRATEGY INSIGHTS ===")
best_value = pricing_analysis.loc[pricing_analysis['value_score'].idxmax()]
lowest_price_per_kwh = pricing_analysis.loc[pricing_analysis['price_per_kwh'].idxmin()]
premium_pricing = pricing_analysis.loc[pricing_analysis['price_avg'].idxmax()]

print(f"💰 Best Value Proposition: {best_value['manufacturer']} {best_value['vehicle_type']} (Value Score: {best_value['value_score']:.2f})")
print(f"⚡ Lowest Price per kWh: {lowest_price_per_kwh['manufacturer']} (${lowest_price_per_kwh['price_per_kwh']:.2f}/kWh)")
print(f"💎 Premium Pricing: {premium_pricing['manufacturer']} {premium_pricing['vehicle_type']} (${premium_pricing['price_avg']:,.0f})")

# Price elasticity insights
print(f"\n📊 Price Elasticity Indicators:")
print(f"  Average Price Range: ${pricing_analysis['price_range_width'].mean():,.0f}")
print(f"  Average Price per kWh: ${pricing_analysis['price_per_kwh'].mean():.2f}")
print(f"  Most Price-Sensitive Category: {pricing_analysis.loc[pricing_analysis['sales_amount'].idxmax(), 'vehicle_category']}")

## 4. Infrastructure Impact Analysis

In [ ]:
# Infrastructure impact on EV adoption
infrastructure_analysis = merged_data.groupby('state').agg({
    'sales_amount': 'sum',
    'ev_penetration_rate': 'mean',
    'total_stations': 'mean',
    'stations_per_1000_ev': 'mean',
    'utilization_rate': 'mean',
    'gdp_per_capita': 'mean',
    'incentives_available': lambda x: x.mode().iloc[0] if not x.mode().empty else 0
}).reset_index()

# Calculate infrastructure efficiency metrics
infrastructure_analysis['station_efficiency'] = (
    infrastructure_analysis['sales_amount'] / infrastructure_analysis['total_stations']
)

infrastructure_analysis['infrastructure_score'] = (
    infrastructure_analysis['stations_per_1000_ev'] * 
    infrastructure_analysis['utilization_rate'] * 
    (1 + infrastructure_analysis['incentives_available'])
)

# Create infrastructure impact dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Infrastructure vs EV Adoption', 
                   'Station Efficiency Analysis',
                   'Infrastructure Score Ranking', 'Incentives Impact'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "bar"}, {"type": "box"}]]
)

# Infrastructure vs EV Adoption
fig.add_trace(
    go.Scatter(
        x=infrastructure_analysis['stations_per_1000_ev'],
        y=infrastructure_analysis['ev_penetration_rate'],
        mode='markers+text',
        text=infrastructure_analysis['state'],
        textposition="top center",
        marker=dict(
            size=infrastructure_analysis['sales_amount']/200,
            color=infrastructure_analysis['gdp_per_capita'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="GDP per Capita")
        ),
        name='Infrastructure Impact'
    ),
    row=1, col=1
)

# Station Efficiency Analysis
fig.add_trace(
    go.Scatter(
        x=infrastructure_analysis['total_stations'],
        y=infrastructure_analysis['station_efficiency'],
        mode='markers+text',
        text=infrastructure_analysis['state'],
        textposition="top center",
        marker=dict(
            size=infrastructure_analysis['utilization_rate']*50,
            color='#ff7f0e'
        ),
        name='Station Efficiency',
        showlegend=False
    ),
    row=1, col=2
)

# Infrastructure Score Ranking
top_infrastructure = infrastructure_analysis.nlargest(10, 'infrastructure_score')
fig.add_trace(
    go.Bar(
        x=top_infrastructure['state'],
        y=top_infrastructure['infrastructure_score'],
        name='Infrastructure Score',
        marker_color='#2ca02c'
    ),
    row=2, col=1
)

# Incentives Impact
incentives_yes = infrastructure_analysis[infrastructure_analysis['incentives_available'] == 1]
incentives_no = infrastructure_analysis[infrastructure_analysis['incentives_available'] == 0]

fig.add_trace(
    go.Box(
        y=incentives_yes['ev_penetration_rate'],
        name='With Incentives',
        marker_color='#1f77b4'
    ),
    row=2, col=2
)

fig.add_trace(
    go.Box(
        y=incentives_no['ev_penetration_rate'],
        name='Without Incentives',
        marker_color='#d62728'
    ),
    row=2, col=2
)

fig.update_layout(height=800, title_text="Charging Infrastructure Impact Analysis")
fig.show()

# Infrastructure insights
print("=== INFRASTRUCTURE IMPACT INSIGHTS ===")
best_infrastructure = infrastructure_analysis.loc[infrastructure_analysis['infrastructure_score'].idxmax()]
most_efficient = infrastructure_analysis.loc[infrastructure_analysis['station_efficiency'].idxmax()]

print(f"🏆 Best Infrastructure: {best_infrastructure['state']} (Score: {best_infrastructure['infrastructure_score']:.2f})")
print(f"⚡ Most Efficient: {most_efficient['state']} ({most_efficient['station_efficiency']:.2f} EVs per station)")
print(f"📈 Incentives Impact: States with incentives have {incentives_yes['ev_penetration_rate'].mean():.2%} avg penetration vs {incentives_no['ev_penetration_rate'].mean():.2%} without")

# Infrastructure recommendations
print(f"\n🎯 Infrastructure Recommendations:")
under_served = infrastructure_analysis[infrastructure_analysis['stations_per_1000_ev'] < 1.0]
print(f"  Under-served states: {', '.join(under_served['state'].tolist())}")
print(f"  Target stations per 1000 EVs: {infrastructure_analysis['stations_per_1000_ev'].mean():.2f}")

## 5. Market Opportunity Analysis

In [ ]:
# Market opportunity scoring
opportunity_analysis = merged_data.groupby('state').agg({
    'sales_amount': 'sum',
    'ev_penetration_rate': 'mean',
    'gdp_per_capita': 'mean',
    'total_population': 'mean',
    'total_stations': 'mean',
    'market_potential_score': 'mean',
    'gasoline_price': 'mean',
    'incentives_available': lambda x: x.mode().iloc[0] if not x.mode().empty else 0
}).reset_index()

# Calculate opportunity metrics
opportunity_analysis['market_saturation'] = opportunity_analysis['ev_penetration_rate']
opportunity_analysis['growth_potential'] = (
    (1 - opportunity_analysis['market_saturation']) * 
    opportunity_analysis['gdp_per_capita'] / 1000 * 
    (1 + opportunity_analysis['incentives_available']) *
    opportunity_analysis['gasoline_price']
)

opportunity_analysis['infrastructure_gap'] = (
    2.0 - opportunity_analysis['total_stations'] / opportunity_analysis['sales_amount'] * 1000
).clip(lower=0)

# Create opportunity score
scaler = StandardScaler()
opportunity_features = ['growth_potential', 'market_potential_score', 'infrastructure_gap', 'gdp_per_capita']
opportunity_analysis['opportunity_score'] = scaler.fit_transform(
    opportunity_analysis[opportunity_features]
).sum(axis=1)

# Create opportunity analysis dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Market Opportunity Matrix', 
                   'Growth Potential vs Saturation',
                   'Top Opportunity States', 'Infrastructure Gap Analysis'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Market Opportunity Matrix
fig.add_trace(
    go.Scatter(
        x=opportunity_analysis['market_saturation'],
        y=opportunity_analysis['growth_potential'],
        mode='markers+text',
        text=opportunity_analysis['state'],
        textposition="top center",
        marker=dict(
            size=opportunity_analysis['opportunity_score']*5 + 10,
            color=opportunity_analysis['opportunity_score'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Opportunity Score")
        ),
        name='Market Opportunity'
    ),
    row=1, col=1
)

# Growth Potential vs Saturation
fig.add_trace(
    go.Scatter(
        x=opportunity_analysis['market_saturation'],
        y=opportunity_analysis['growth_potential'],
        mode='markers',
        marker=dict(
            size=opportunity_analysis['total_population']/100000,
            color=opportunity_analysis['incentives_available'],
            colorscale='RdYlBu',
            showscale=True,
            colorbar=dict(title="Incentives Available")
        ),
        name='Growth vs Saturation',
        showlegend=False
    ),
    row=1, col=2
)

# Top Opportunity States
top_opportunities = opportunity_analysis.nlargest(10, 'opportunity_score')
fig.add_trace(
    go.Bar(
        x=top_opportunities['state'],
        y=top_opportunities['opportunity_score'],
        name='Opportunity Score',
        marker_color='#2ca02c'
    ),
    row=2, col=1
)

# Infrastructure Gap Analysis
infrastructure_gap_states = opportunity_analysis.nlargest(10, 'infrastructure_gap')
fig.add_trace(
    go.Bar(
        x=infrastructure_gap_states['state'],
        y=infrastructure_gap_states['infrastructure_gap'],
        name='Infrastructure Gap',
        marker_color='#ff7f0e'
    ),
    row=2, col=2
)

fig.update_layout(height=800, title_text="EV Market Opportunity Analysis")
fig.show()

# Opportunity insights
print("=== MARKET OPPORTUNITY INSIGHTS ===")
top_opportunity = opportunity_analysis.loc[opportunity_analysis['opportunity_score'].idxmax()]
highest_growth = opportunity_analysis.loc[opportunity_analysis['growth_potential'].idxmax()]
biggest_gap = opportunity_analysis.loc[opportunity_analysis['infrastructure_gap'].idxmax()]

print(f"🎯 Top Opportunity: {top_opportunity['state']} (Score: {top_opportunity['opportunity_score']:.2f})")
print(f"📈 Highest Growth Potential: {highest_growth['state']} (Potential: {highest_growth['growth_potential']:.2f})")
print(f"🏗️ Biggest Infrastructure Gap: {biggest_gap['state']} (Gap: {biggest_gap['infrastructure_gap']:.2f})")

# Market recommendations
print(f"\n💡 STRATEGIC RECOMMENDATIONS:")
print(f"  Priority Markets: {', '.join(top_opportunities.head(3)['state'].tolist())}")
print(f"  Infrastructure Investment: {', '.join(infrastructure_gap_states.head(3)['state'].tolist())}")
print(f"  Incentive Programs: Focus on states with low saturation but high potential")

## 6. Strategic Business Intelligence

In [ ]:
# Generate comprehensive business intelligence summary
business_intelligence = {
    'market_overview': {
        'total_sales': int(ev_sales['sales_amount'].sum()),
        'total_revenue': float(ev_sales['revenue'].sum()),
        'avg_penetration': float(market_metrics['ev_penetration_rate'].mean()),
        'market_segments': state_features['market_segment'].value_counts().to_dict(),
        'top_states': state_features.nlargest(3, 'sales_amount')['state'].tolist()
    },
    'competitive_insights': {
        'market_leader': market_leader['manufacturer'],
        'market_share_percentage': float(market_leader['market_share_pct']),
        'technology_leader': tech_leader['manufacturer'],
        'premium_brand': premium_brand['manufacturer'],
        'competitive_intensity': len(manufacturer_competitive)
    },
    'pricing_intelligence': {
        'avg_vehicle_price': float(ev_sales['price_avg'].mean()),
        'price_range': {
            'economy': float(ev_sales[ev_sales['price_avg'] < 40000]['price_avg'].mean()),
            'mid_range': float(ev_sales[(ev_sales['price_avg'] >= 40000) & (ev_sales['price_avg'] < 70000)]['price_avg'].mean()),
            'premium': float(ev_sales[ev_sales['price_avg'] >= 70000]['price_avg'].mean())
        },
        'best_value_proposition': best_value['manufacturer'] + ' ' + best_value['vehicle_type'],
        'price_elasticity_indicators': {
            'avg_price_per_kwh': float(pricing_analysis['price_per_kwh'].mean()),
            'value_score_range': [float(pricing_analysis['value_score'].min()), float(pricing_analysis['value_score'].max())]
        }
    },
    'infrastructure_analysis': {
        'total_charging_stations': int(charging_stations['total_stations'].sum()),
        'avg_utilization_rate': float(charging_stations['utilization_rate'].mean()),
        'best_infrastructure_state': best_infrastructure['state'],
        'infrastructure_gap_states': under_served['state'].tolist(),
        'incentives_impact': {
            'with_incentives_penetration': float(incentives_yes['ev_penetration_rate'].mean()),
            'without_incentives_penetration': float(incentives_no['ev_penetration_rate'].mean())
        }
    },
    'opportunity_analysis': {
        'top_opportunity_state': top_opportunity['state'],
        'opportunity_score': float(top_opportunity['opportunity_score']),
        'priority_markets': top_opportunities.head(5)['state'].tolist(),
        'infrastructure_investment_needs': infrastructure_gap_states.head(5)['state'].tolist(),
        'growth_potential_leaders': opportunity_analysis.nlargest(3, 'growth_potential')['state'].tolist()
    },
    'strategic_recommendations': {
        'market_expansion': f"Focus on {', '.join(top_opportunities.head(3)['state'].tolist())} for highest ROI",
        'infrastructure_development': f"Prioritize {', '.join(infrastructure_gap_states.head(3)['state'].tolist())} for station deployment",
        'competitive_positioning': f"Compete on {best_value['manufacturer']}'s value proposition and {tech_leader['manufacturer']}'s technology",
        'pricing_strategy': f"Target mid-range segment (${ev_sales[(ev_sales['price_avg'] >= 40000) & (ev_sales['price_avg'] < 70000)]['price_avg'].mean():,.0f}) for balanced growth",
        'policy_advocacy': "Push for incentives in under-penetrated high-potential markets"
    }
}

# Save business intelligence report
import json
with open('../reports/market_analysis_insights.json', 'w') as f:
    json.dump(business_intelligence, f, indent=2, default=str)

# Print executive summary
print("=== EXECUTIVE BUSINESS INTELLIGENCE SUMMARY ===")
print(f"📊 Market Size: {business_intelligence['market_overview']['total_sales']:,} EVs (${business_intelligence['market_overview']['total_revenue']:,.2f} revenue)")
print(f"🏆 Market Leader: {business_intelligence['competitive_insights']['market_leader']} ({business_intelligence['competitive_insights']['market_share_percentage']:.1f}% share)")
print(f"💎 Average Price Point: ${business_intelligence['pricing_intelligence']['avg_vehicle_price']:,.0f}")
print(f"⚡ Infrastructure Coverage: {business_intelligence['infrastructure_analysis']['total_charging_stations']:,} stations")
print(f"🎯 Top Opportunity: {business_intelligence['opportunity_analysis']['top_opportunity_state']}")

print(f"\n🚀 KEY STRATEGIC INITIATIVES:")
for initiative, description in business_intelligence['strategic_recommendations'].items():
    print(f"  • {initiative.replace('_', ' ').title()}: {description}")

print(f"\n✅ Market analysis completed! Business intelligence saved to reports/market_analysis_insights.json")